In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Load and split the breast cancer dataset
cancer = load_breast_cancer()
X, y   = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Classes:', cancer.target_names)
print('Train shape:', X_train.shape)
print('Test  shape:', X_test.shape)

In [ ]:
# Sigmoid and binary cross-entropy

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def bce_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

z = np.linspace(-6, 6, 200)
plt.figure(figsize=(6, 3))
plt.plot(z, sigmoid(z), linewidth=2)
plt.axhline(0.5, color='red', linestyle='--', label='decision boundary')
plt.title('Sigmoid Activation'); plt.xlabel('z'); plt.ylabel('σ(z)')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Gradient descent training

n, d  = X_train.shape
W     = np.zeros(d)
b     = 0.0
lr    = 0.1
epochs = 1000
losses = []

for epoch in range(epochs):
    y_hat = sigmoid(X_train @ W + b)
    losses.append(bce_loss(y_train, y_hat))

    dW = (X_train.T @ (y_hat - y_train)) / n
    db = np.mean(y_hat - y_train)
    W -= lr * dW
    b -= lr * db

print('Final loss:', round(losses[-1], 4))

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.title('Training Loss (BCE)'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.tight_layout(); plt.show()

In [ ]:
# Predictions and accuracy

def predict(X, W, b, threshold=0.5):
    return (sigmoid(X @ W + b) >= threshold).astype(int)

y_pred_train = predict(X_train, W, b)
y_pred_test  = predict(X_test,  W, b)

train_acc = np.mean(y_pred_train == y_train)
test_acc  = np.mean(y_pred_test  == y_test)

print(f'Train Accuracy : {train_acc:.4f}')
print(f'Test  Accuracy : {test_acc:.4f}')

In [ ]:
# Confusion matrix

cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks([0, 1], cancer.target_names, rotation=15)
plt.yticks([0, 1], cancer.target_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=14,
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout(); plt.show()

print(classification_report(y_test, y_pred_test, target_names=cancer.target_names))

In [ ]:
# Top 10 most influential features (by absolute weight)

feat_names  = cancer.feature_names
idx_sorted  = np.argsort(np.abs(W))[::-1][:10]

plt.figure(figsize=(9, 4))
colors = ['steelblue' if W[i] >= 0 else 'tomato' for i in idx_sorted]
plt.bar(range(10), W[idx_sorted], color=colors, edgecolor='black')
plt.xticks(range(10), feat_names[idx_sorted], rotation=40, ha='right')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Top 10 Feature Weights (blue = positive, red = negative)')
plt.ylabel('Weight')
plt.tight_layout(); plt.show()